In [1]:
import os
import json
import shutil
from PIL import Image
from tqdm import tqdm

Image.MAX_IMAGE_PIXELS = None  # Disable the limit on image size

In [2]:
# --- CONFIGURATION ---
# 1. Path to the root of your fMoW dataset (the folder containing class folders like 'surface_mine/')
FMoW_ROOT_PATH = '/caa/Homes01/mburges/datasets/fMoW_original' 

# 2. Path to the folder where you want to save all the images
OUTPUT_IMAGE_DIR = '/caa/Homes01/mburges/datasets/fMoW/images'

# 3. Path where you want to save the final COCO JSON file
OUTPUT_JSON_PATH = '/caa/Homes01/mburges/datasets/fMoW/annotations/fmow_coco.json'
# --- END CONFIGURATION ---


def convert_fmow_to_coco(root_path, output_image_dir, output_json_path):
    """
    Converts the fMoW-rgb dataset to COCO object detection format.
    """
    # Create the output directory for images if it doesn't exist
    os.makedirs(output_image_dir, exist_ok=True)
    # Ensure the output JSON directory exists
    os.makedirs(os.path.dirname(output_json_path), exist_ok=True)

    # Initialize the main COCO structure
    coco_output = {
        "info": {
            "description": "fMoW-rgb Dataset in COCO format",
            "version": "1.0",
            "year": 2024,
            "contributor": "fMoW Dataset and custom script",
            "date_created": "" # Will be filled later
        },
        "licenses": [],
        "images": [],
        "annotations": [],
        "categories": []
    }

    # Initialize counters and mappings
    image_id_counter = 1
    annotation_id_counter = 1
    category_map = {} # Maps class name to category ID

    # Get a sorted list of class names from the folder structure
    class_names = sorted([d for d in os.listdir(root_path) if os.path.isdir(os.path.join(root_path, d))])

    print(f"Found {len(class_names)} classes. Starting conversion...")

    # --- 1. Create Categories ---
    for i, class_name in enumerate(class_names):
        category_id = i + 1  # COCO category IDs start from 1
        category_map[class_name] = category_id
        coco_output["categories"].append({
            "id": category_id,
            "name": class_name,
            "supercategory": class_name
        })

    print("Categories created successfully.")

    # --- 2. Process Images and Annotations ---
    # Use tqdm for a progress bar over the classes
    for class_name in tqdm(class_names, desc="Processing classes"):
        category_id = category_map[class_name]
        class_path = os.path.join(root_path, class_name)

        for location_folder in os.listdir(class_path):
            location_path = os.path.join(class_path, location_folder)
            if not os.path.isdir(location_path):
                continue
            
            # Process each file pair in the location folder
            for filename in os.listdir(location_path):
                # We only process based on the json file for RGB images
                if not filename.endswith('_rgb.json'):
                    continue

                json_path = os.path.join(location_path, filename)
                # Corresponding image is the same name with .jpg extension
                image_filename = filename.replace('.json', '.jpg')
                image_path = os.path.join(location_path, image_filename)

                if not os.path.exists(image_path):
                    print(f"Warning: Image file not found for {json_path}")
                    continue

                try:
                    # --- Read Metadata and Image Info ---
                    with open(json_path, 'r') as f:
                        metadata = json.load(f)

                    # Get image dimensions
                    with Image.open(image_path) as img:
                        width, height = img.size

                    # --- Copy Image to Output Folder ---
                    shutil.copy(image_path, os.path.join(output_image_dir, image_filename))

                    # --- Add Image Entry to COCO JSON ---
                    image_info = {
                        "id": image_id_counter,
                        "file_name": image_filename,
                        "width": width,
                        "height": height,
                        "license": None,
                        "flickr_url": None,
                        "coco_url": None,
                        "date_captured": metadata.get("timestamp", "")
                    }
                    coco_output["images"].append(image_info)

                    # --- Add Annotation Entries to COCO JSON ---
                    if "bounding_boxes" in metadata:
                        for bbox_data in metadata["bounding_boxes"]:
                            # fMoW format [x_min, y_min, width, height] is the same as COCO
                            bbox = bbox_data["box"]
                            
                            # Ensure bbox is valid
                            if bbox[2] <= 0 or bbox[3] <= 0:
                                continue

                            annotation_info = {
                                "id": annotation_id_counter,
                                "image_id": image_id_counter,
                                "category_id": category_id, # From the folder structure
                                "bbox": bbox,
                                "area": bbox[2] * bbox[3],
                                "segmentation": [], # Not provided in fMoW
                                "iscrowd": 0
                            }
                            coco_output["annotations"].append(annotation_info)
                            annotation_id_counter += 1
                    
                    # Increment image ID for the next unique image
                    image_id_counter += 1

                except Exception as e:
                    print(f"Error processing file {json_path}: {e}")

    # Set creation date
    from datetime import datetime
    coco_output["info"]["date_created"] = datetime.now().strftime("%Y-%m-%d")

    # --- 3. Save the final COCO JSON file ---
    print(f"\nConversion complete. Saving COCO JSON to {output_json_path}")
    with open(output_json_path, 'w') as f:
        json.dump(coco_output, f, indent=4)

    print("Done! 🎉")
    print(f"Total images: {len(coco_output['images'])}")
    print(f"Total annotations: {len(coco_output['annotations'])}")


if __name__ == '__main__':
    convert_fmow_to_coco(FMoW_ROOT_PATH, OUTPUT_IMAGE_DIR, OUTPUT_JSON_PATH)


Found 62 classes. Starting conversion...
Categories created successfully.


Processing classes: 100%|██████████| 62/62 [00:24<00:00,  2.49it/s]



Conversion complete. Saving COCO JSON to /caa/Homes01/mburges/datasets/fMoW/annotations/fmow_coco.json
Done! 🎉
Total images: 53041
Total annotations: 53041
